                         ┌──────────────────────┐
                         │  pipeline_metadata   │
                         │ "WHAT should run?"   │
                         └──────────┬───────────┘
                                    │
                                    ▼
                         Generic Pipeline Engine
                                    │
             ┌──────────────────────┼─────────────────────┐
             │                      │                     │
             ▼                      ▼                     ▼
        Bronze Task            Silver Task            Gold Task
             │                      │                     │
             └──────────────────────┼─────────────────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │  pipeline_run_log    │
                         │ "WHAT happened?"     │
                         └──────────────────────┘
                                    │
                 ┌──────────────────┼──────────────────┐
                 ▼                  ▼                  ▼
        pipeline_task_log    data_quality_log   pipeline_error_log
                                    │
                                    ▼
                              quarantine_log

In [0]:
# Import the current_timestamp function.
# We will use it to record when the metadata row was created and updated.
from pyspark.sql.functions import current_timestamp

# Import Spark data types.
# These are used to explicitly define the structure of our metadata table.
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    TimestampType
)

# ---------------------------------------------------------
# 1. DEFINE THE METADATA TABLE SCHEMA
# ---------------------------------------------------------

# StructType represents the complete schema of our DataFrame.
# Think of it as the definition of all columns in our metadata table.
metadata_schema = StructType([

    # Unique identifier for this pipeline configuration.
    StructField("pipeline_id", StringType(), False),

    # Human-readable pipeline name.
    StructField("pipeline_name", StringType(), False),

    # Name of the source system.
    StructField("source_name", StringType(), False),

    # Location where the source data is stored.
    # In our case, the NYC Taxi Parquet files are in a Unity Catalog Volume.
    StructField("source_path", StringType(), False),

    # File format of the source data.
    # Example: parquet, csv, json, etc.
    StructField("source_format", StringType(), False),

    # Target catalog where the data will be stored.
    StructField("target_catalog", StringType(), False),

    # Target schema where the table will be created.
    StructField("target_schema", StringType(), False),

    # Target table that our generic framework will write to.
    StructField("target_table", StringType(), False),

    # Defines how the pipeline should load data.
    # Examples: FULL or INCREMENTAL.
    StructField("load_type", StringType(), False),

    # Business columns that can be used to identify duplicate records
    # or determine the logical key for a dataset.
    StructField("business_key", StringType(), True),

    # Column used to identify new/changed records during incremental processing.
    StructField("watermark_column", StringType(), True),

    # Name of the Data Quality rule set that should be applied.
    StructField("dq_rule_set", StringType(), True),

    # Indicates whether this pipeline configuration is currently active.
    # True  = framework should process it.
    # False = framework should ignore it.
    StructField("active_flag", BooleanType(), False),

    # Timestamp showing when this metadata configuration was created.
    StructField("created_at", TimestampType(), True),

    # Timestamp showing when this metadata configuration was last modified.
    StructField("updated_at", TimestampType(), True)
])

# ---------------------------------------------------------
# 2. CREATE INITIAL METADATA CONFIGURATION
# ---------------------------------------------------------

# Each tuple represents one configuration record.
# Our generic pipeline framework will eventually read these records
# instead of having source paths and table names hardcoded in Python code.
metadata_data = [

    (
        # Unique pipeline ID.
        "NYC_YELLOW_001",

        # Pipeline name.
        "nyc_taxi_yellow",

        # Source system.
        "NYC_TLC",

        # Root location of our raw NYC Taxi files.
        "/Volumes/workspace/nyc_taxi_bronze/raw_volume/",

        # Source format.
        "parquet",

        # Target catalog.
        "workspace",

        # Target Bronze schema.
        "nyc_taxi_bronze",

        # Target Bronze table.
        "yellow_taxi",

        # We are designing the framework for incremental processing.
        # We will refine the actual incremental logic later.
        "INCREMENTAL",

        # Candidate business key.
        # IMPORTANT:
        # We will validate this against the actual NYC Taxi source data
        # before treating it as a true production business key.
        "VendorID,tpep_pickup_datetime,tpep_dropoff_datetime",

        # Timestamp column we expect to use for incremental processing.
        "tpep_pickup_datetime",

        # Name of the Data Quality rule set.
        "yellow_taxi_default",

        # This configuration is active.
        True,

        # We will populate these timestamps below.
        None,
        None
    )
]

# ---------------------------------------------------------
# 3. CREATE A DATAFRAME FROM THE METADATA
# ---------------------------------------------------------

# Convert our Python list into a Spark DataFrame.
# The explicit metadata_schema ensures that every column
# has the data type we defined above.
metadata_df = spark.createDataFrame(metadata_data,metadata_schema)
# ---------------------------------------------------------
# 4. ADD AUDIT TIMESTAMPS
# ---------------------------------------------------------

# Add the current timestamp as the creation time.
metadata_df = metadata_df.withColumn("created_at",current_timestamp())

# Add the current timestamp as the last-update time.
metadata_df = metadata_df.withColumn("updated_at",current_timestamp())

# ---------------------------------------------------------
# 5. WRITE THE METADATA TABLE AS DELTA
# ---------------------------------------------------------

# Save the DataFrame as a managed Delta table inside our audit schema.
#
# Fully qualified table name:
# workspace.nyc_taxi_audit.pipeline_metadata
#
# "overwrite" is acceptable during this initial setup because
# we are creating the table for the first time.
metadata_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.nyc_taxi_audit.pipeline_metadata"
    )


# ---------------------------------------------------------
# 6. READ THE TABLE BACK AND DISPLAY IT
# ---------------------------------------------------------

# Read the newly created metadata table.
pipeline_metadata_df = spark.table(
    "workspace.nyc_taxi_audit.pipeline_metadata"
)

# Display the contents so we can verify that the table
# was created successfully and the metadata is correct.
display(pipeline_metadata_df)

In [0]:
# ============================================================
# 7. CREATE PIPELINE RUN LOG
# ============================================================

# This table stores one row for every complete pipeline execution.
# It answers:
# "WHAT happened during the pipeline run?"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.nyc_taxi_audit.pipeline_run_log (

    -- Unique execution identifier.
    run_id STRING NOT NULL,

    -- ID of the pipeline configuration.
    pipeline_id STRING NOT NULL,

    -- Human-readable pipeline name.
    pipeline_name STRING NOT NULL,

    -- How the pipeline was started.
    -- Examples: MANUAL, SCHEDULED, WORKFLOW.
    trigger_type STRING,

    -- File or batch being processed.
    source_file STRING,

    -- Pipeline start timestamp.
    run_start_time TIMESTAMP NOT NULL,

    -- Pipeline completion timestamp.
    run_end_time TIMESTAMP,

    -- Overall pipeline status.
    -- Examples: RUNNING, SUCCESS, FAILED, PARTIAL.
    status STRING NOT NULL,

    -- Number of records read from source.
    records_read BIGINT,

    -- Number of records successfully written.
    records_written BIGINT,

    -- Number of rejected records.
    records_rejected BIGINT,

    -- Watermark value used during this run.
    watermark_value STRING,

    -- Error message when the run fails.
    error_message STRING,

    -- Databricks job run identifier.
    job_run_id STRING,

    -- Timestamp when the audit record was created.
    created_at TIMESTAMP NOT NULL
)
USING DELTA
""")


In [0]:
# ============================================================
# 8. CREATE PIPELINE TASK LOG
# ============================================================

# This table stores execution information for individual tasks.
# It answers:
# "WHICH task failed?"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.nyc_taxi_audit.pipeline_task_log (

    -- Overall pipeline execution ID.
    run_id STRING NOT NULL,

    -- Pipeline configuration ID.
    pipeline_id STRING NOT NULL,

    -- Name of the individual task.
    -- Example: bronze_ingestion, silver_transform.
    task_name STRING NOT NULL,

    -- Execution order of the task.
    task_sequence INT,

    -- Task start timestamp.
    start_time TIMESTAMP NOT NULL,

    -- Task completion timestamp.
    end_time TIMESTAMP,

    -- Current task status.
    -- Examples: RUNNING, SUCCESS, FAILED, SKIPPED.
    status STRING NOT NULL,

    -- Number of records processed by this task.
    records_processed BIGINT,

    -- Error message for the task.
    error_message STRING,

    -- Databricks task run ID.
    task_run_id STRING,

    -- Audit record creation timestamp.
    created_at TIMESTAMP NOT NULL
)
USING DELTA
""")


In [0]:
# ============================================================
# 9. CREATE DATA QUALITY LOG
# ============================================================

# This table stores the results of data quality rules.
# It answers:
# "WHICH data quality rules passed or failed?"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.nyc_taxi_audit.data_quality_log (

    -- Pipeline execution ID.
    run_id STRING NOT NULL,

    -- Pipeline configuration ID.
    pipeline_id STRING NOT NULL,

    -- Table where the quality rule was applied.
    table_name STRING NOT NULL,

    -- Name of the quality rule.
    rule_name STRING NOT NULL,

    -- Type/category of the quality rule.
    -- Examples: NULL_CHECK, RANGE_CHECK, DUPLICATE_CHECK.
    rule_type STRING,

    -- Number of records checked.
    records_checked BIGINT,

    -- Number of valid records.
    records_passed BIGINT,

    -- Number of invalid records.
    records_failed BIGINT,

    -- Overall result of the rule.
    -- Expected values: PASS or FAIL.
    status STRING NOT NULL,

    -- Business explanation of the rule.
    rule_description STRING,

    -- Timestamp when the rule was executed.
    execution_time TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
# ============================================================
# 10. CREATE QUARANTINE LOG
# ============================================================

# This table stores metadata about records rejected by DQ rules.
# The actual rejected data can be stored separately.
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.nyc_taxi_audit.quarantine_log (

    -- Pipeline execution ID.
    run_id STRING NOT NULL,

    -- Pipeline configuration ID.
    pipeline_id STRING NOT NULL,

    -- Source file containing the bad record.
    source_file STRING,

    -- Identifier or hash representing the record.
    record_id STRING,

    -- Name of the failed DQ rule.
    failed_rule STRING NOT NULL,

    -- Explanation of why the record was rejected.
    failure_reason STRING NOT NULL,

    -- Source table or processing stage.
    source_table STRING,

    -- Timestamp when the record was quarantined.
    quarantine_time TIMESTAMP NOT NULL
)
USING DELTA
""")


In [0]:
# ============================================================
# 11. CREATE PIPELINE ERROR LOG
# ============================================================

# This table stores technical/application errors.
# It is different from data quality failures.
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.nyc_taxi_audit.pipeline_error_log (

    -- Pipeline execution ID.
    run_id STRING NOT NULL,

    -- Pipeline configuration ID.
    pipeline_id STRING NOT NULL,

    -- Task where the technical error occurred.
    task_name STRING,

    -- Category of the technical error.
    -- Examples: SCHEMA_ERROR, RUNTIME_ERROR, PERMISSION_ERROR.
    error_type STRING,

    -- Actual error message.
    error_message STRING NOT NULL,

    -- Detailed stack trace when available.
    stack_trace STRING,

    -- Timestamp of the technical error.
    error_time TIMESTAMP NOT NULL
)
USING DELTA
""")


In [0]:
# ============================================================
# 12. VERIFY ALL AUDIT TABLES
# ============================================================

# SHOW TABLES returns all tables in our audit schema.
audit_tables_df = spark.sql("""
    SHOW TABLES IN workspace.nyc_taxi_audit
""")

# Display the tables so we can verify the audit layer.
display(audit_tables_df)

In [0]:
# ============================================================
# CELL 7 — CREATE FILE INGESTION LOG
# ============================================================

# Create a dedicated audit table for source-file processing.
#
# This table is separate from pipeline_run_log because:
#
# pipeline_run_log
#     = one record per pipeline execution
#
# file_ingestion_log
#     = one record per source file processed
#
# This separation gives us better operational visibility
# when one pipeline run processes multiple files.
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.nyc_taxi_audit.file_ingestion_log (

    -- Unique identifier for the pipeline execution.
    run_id STRING NOT NULL,

    -- Pipeline configuration identifier.
    pipeline_id STRING NOT NULL,

    -- Name of the pipeline.
    pipeline_name STRING NOT NULL,

    -- Complete source-file path.
    source_file STRING NOT NULL,

    -- Source-file name without the directory path.
    source_file_name STRING,

    -- File size in bytes when discovered.
    file_size_bytes BIGINT,

    -- Time when the file was discovered.
    discovered_at TIMESTAMP NOT NULL,

    -- Time when processing of the file started.
    processing_start_time TIMESTAMP,

    -- Time when processing of the file finished.
    processing_end_time TIMESTAMP,

    -- File-level processing status.
    --
    -- Expected values:
    -- DISCOVERED
    -- PROCESSING
    -- SUCCESS
    -- FAILED
    -- SKIPPED
    status STRING NOT NULL,

    -- Number of records read from the file.
    records_read BIGINT,

    -- Number of records written to Bronze.
    records_written BIGINT,

    -- Number of records rejected.
    records_rejected BIGINT,

    -- Error message if file processing fails.
    error_message STRING,

    -- Timestamp when the audit record was created.
    created_at TIMESTAMP NOT NULL
)
USING DELTA
""")


# Confirm that the file-level audit table exists.
print(
    "file_ingestion_log table created successfully."
)